## Middleware

Middleware provides a way to more tightly control what happens inside the agents. Middleware is useful for the following.

- Tracking agent behaviour with logging, analytics and debugging.
- Transforming prompts, tool selections, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails and Pil detection.

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

## Summarization Middleware

Automatically Summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following.

- Long-running conversations that exceed context windows.
- Multi-turn dialogues  with extensive history.
- Application where preserving full conversation context matters.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based Summarization
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [3]:
# Run with thread ID
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# Alternative test data
questions=[
    "What is 2+2?",
    "What is 10/2?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4-4?",
    "What is 9-5?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='07273d75-3b6e-42cc-be6f-1e3a22b2aa3f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f131a-51dc-7c81-8c60-2caf46bb9595-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 18}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='07273d75-3b6e-42cc-be6f-1e3a22b2aa3f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f131a-51dc-7c81-8c60-2caf46bb9595-0', tool_calls

### Token Size

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hotels(city:str)->str:
    """Search hotels- return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350 night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness centre
    3. Budget stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

# Token counter (approximate)
def count_token(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars //4   # 4 chars = 1 token

In [ ]:
# Run test
cities=["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},config=config
    )

    tokens=count_token(response["messages"])
    print(f"{city}: ~ {tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response["messages"])}")

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core import tools
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str)->str:
    """Search hotels"""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget stay $75"

# Low fraction for testing!
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("fraction",0.005),  # 0.005 ~ 640 tokens
            keep=("fraction", 0.002)     # 0.002 ~ 256 tokens
        )
    ]
)

config={"configurable": {"thread_id":"test-1"}}

# Token Counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)//4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},config=config
    )

    tokens=count_token(response["messages"])
    fractions = tokens/128000
    print(f"{city}: ~ {tokens} tokens ({fractions:.4%}), {len(response['messages'])} msgs")
    print(f"{(response["messages"])}")
    

## Human In Loop Middleware

Pause agent execution for human approval, aditing or rejection of tool call before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval(eg: database wires, final transactions).
- Compilance workflow where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipent:str, subject:str, body:str)->str:
    """Mock function to send an email"""
    return f"Email send to {recipent} with subject {subject}"

In [18]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "test-approve"
    }
}

#Step: 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to harshjyoriya@gmail.com with subject hello and the body how are you?"
            )
        ]
    },
    config=config
)


In [22]:
result

{'messages': [HumanMessage(content='Send email to harshjyoriya@gmail.com with subject hello and the body how are you?', additional_kwargs={}, response_metadata={}, id='f8ef1d32-30e9-49d9-a061-856454e65f99'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipent": "harshjyoriya@gmail.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'44f68437-231a-42c9-986e-9dd966c543c9': 'CoICAQw51scxFwFuILiXv5WktkBmiMjDitcWwDdNByF2hxshjBfTmKxFbb4aZJTocdvnFckKKKjM/+SARIhy1ACQNGlydwJHv15h85xCCPF+/Iw6pVlsM8479U0wcf+6JOji8RK1JT37ekLTJyx1mzvnAienBeZM8Ao35r2kLRFpk2dmgvAk2rYdg8+UgfL1CM2NoId32vKEbudMCOrD5laEBxta7pnzBT0vJmSwBT2mFpdCLCkx4oGs1PdJ0D7VYMFyZm6PfhzjPpzrVHur1v3UkjYZWKW/qCbfGmuc71LSg1a6c2IFdSFk4fxlSSKbZ5JcLsms7XBVeONAAE85+IqHNDC1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f13a2-a07d-7

In [24]:
from langgraph.types import Command

# Step 2: Approve

if "__interrupt__" in result:
    print("⏸️ Paused Approving")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused Approving
✅ Result: I have sent the email to harshjyoriya@gmail.com with subject hello and body how are you?


In [25]:
result

{'messages': [HumanMessage(content='Send email to harshjyoriya@gmail.com with subject hello and the body how are you?', additional_kwargs={}, response_metadata={}, id='f8ef1d32-30e9-49d9-a061-856454e65f99'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipent": "harshjyoriya@gmail.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'44f68437-231a-42c9-986e-9dd966c543c9': 'CoICAQw51scxFwFuILiXv5WktkBmiMjDitcWwDdNByF2hxshjBfTmKxFbb4aZJTocdvnFckKKKjM/+SARIhy1ACQNGlydwJHv15h85xCCPF+/Iw6pVlsM8479U0wcf+6JOji8RK1JT37ekLTJyx1mzvnAienBeZM8Ao35r2kLRFpk2dmgvAk2rYdg8+UgfL1CM2NoId32vKEbudMCOrD5laEBxta7pnzBT0vJmSwBT2mFpdCLCkx4oGs1PdJ0D7VYMFyZm6PfhzjPpzrVHur1v3UkjYZWKW/qCbfGmuc71LSg1a6c2IFdSFk4fxlSSKbZ5JcLsms7XBVeONAAE85+IqHNDC1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f13a2-a07d-7

### Reject

In [26]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [28]:
config = {
    "configurable": {
        "thread_id": "test-reject"
    }
}

#Step: 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to harshjyoriya@gmail.com with subject hello and the body how are you?"
            )
        ]
    },
    config=config
)


In [29]:
from langgraph.types import Command

# Step 2: Approve

if "__interrupt__" in result:
    print("⏸️ Paused Approving")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused Approving
✅ Result: I am sorry, I cannot send emails. I am a large language model, able to communicate in response to a wide range of prompts and questions, but my capabilities do not include sending emails.


### Editing

In [40]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [41]:
config = {
    "configurable": {
        "thread_id": "test-edit"
    }
}

#Step: 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to harshjyoriya@gmail.com with subject hello and the body how are you?"
            )
        ]
    },
    config=config
)


In [42]:
result

{'messages': [HumanMessage(content='Send email to harshjyoriya@gmail.com with subject hello and the body how are you?', additional_kwargs={}, response_metadata={}, id='5a009564-c3a0-492b-a5f6-290638fef8f8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "hello", "recipent": "harshjyoriya@gmail.com", "body": "how are you?"}'}, '__gemini_function_call_thought_signatures__': {'90ae36ae-34bc-4c3b-9f30-6eca6c0c457f': 'Co8CAQw51scIB/VYTNVBI7k1SACnPHOCHLdY5nx4PPNJTW14ycpEZXgR4+71UCQGk/sS5JGnYDfPpWBpEjxhr91365gwa3z3dueccAkh+6EsdrOMQiA1SUZHVB2b1hCtR3+fvoJv8nfRv5TAfFIxQA/GzYmSFfzs6wmvcm59jzmUiRcwJF9EeO7A1pK9vGGsfRL8i5C3Vksqa6tL9pyrDrlK7du64KQM8avBHQDPySZb/nKtF7CSP2mDu21C9HzHiJq/zXiOjgbC5PFK/sG3E677veqE7SY1MvW5PUncxV2PDZkaXO6ynqTTYp7fLI4Ep9xeGqP7SfCCyb1X60ulCQEeoccWfpgdvvFLbmIe3WaH5w=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_

In [ ]:
# Step 2: Edit and approve

if "__interrupt__" in result:
    print("⏸️ Paused Approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                    "type":"edit",
                    "edited_action":{
                        "name":"send_email_tool",   # Tool name
                        "args":{                    # New arguments
                            "recipient":"harshjyoriya639@gmail.com",
                            "subject":"Verifying you",
                            "body":"Have you eaten a food"
                        }
                    }
                    }
                ]
            }
        ),
        config=config,
    )

In [ ]:
result